walkthrough for docs

In [1]:
from pathlib import Path
from metasmith.python_api import *
from metasmith import examples

from local.constants import WORKSPACE_ROOT

In [2]:
EXAMPLES_DIR = WORKSPACE_ROOT/"src/metasmith/example_resources"
dtypes = DataTypeLibrary.Load(EXAMPLES_DIR/"types/minimal_genomics.yml")

xgdb_path = EXAMPLES_DIR/"data/fosmid.xgdb"
xgdb = DataInstanceLibrary.Load(xgdb_path)
# xgdb = DataInstanceLibrary(xgdb_path)
# xgdb.AddTypeLibrary("genomics", dtypes)
# xgdb.Add([
#     (EXAMPLES_DIR/"fosmid.fna", "./fosmid.fna", "genomics::contigs"),
# ])
# xgdb.PruneTypes()
# xgdb.Save()

refdb_path = EXAMPLES_DIR/"data/references.xgdb"
refdb = DataInstanceLibrary.Load(refdb_path)
# refdb = DataInstanceLibrary(refdb_path)
# refdb.AddTypeLibrary("genomics", dtypes)
# refdb.Add([
#     (EXAMPLES_DIR/"swissprot_bcaa.fna", "./swissprot_bcaa.fna", "genomics::aa_sequences"),
# ])
# refdb.PruneTypes()
# refdb.Save()

trans_path = EXAMPLES_DIR/"transforms/gene_annotation"
transforms = TransformInstanceLibrary.Load(trans_path); # transforms.PruneTypes()

# transforms = TransformInstanceLibrary(trans_path)
# transforms.AddTypeLibrary("genomics", dtypes)
# transforms.AddStub("pprodigal")
# transforms.AddStub("diamond")
# transforms.AddStub("make_diamond_db")
# transforms.Save()

In [3]:
agent = Agent(
    home = Source.FromLocal(Path("./cache/local_home").resolve()),
    # home = Source.FromLocal((WORKSPACE_ROOT/"docs/source/metasmith_home").resolve()),
)
agent.Deploy()

2025-03-21_10-37-26  | >>> AGENT_HOME=/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home
2025-03-21_10-37-26  | >>> mkdir -p $AGENT_HOME
2025-03-21_10-37-26  | >>> mkdir -p /home/tony/.globus
2025-03-21_10-37-26  | >>> mkdir -p /home/tony/.globusonline
2025-03-21_10-37-26  | >>> {if not exists}: apptainer pull/metasmith.sif docker://quay.io/hallamlab/metasmith:latest
2025-03-21_10-37-26  | staged [msm_stub]
2025-03-21_10-37-26  | staged [msm]
2025-03-21_10-37-26  | staged [lib/agent.yml]
2025-03-21_10-37-26  | staged [lib/msm_bootstrap]
2025-03-21_10-37-26  | staged [lib/nextflow_config]
2025-03-21_10-37-26  | deploying [5] staged files
2025-03-21_10-37-26  | >>> cd /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home && ./msm api deploy_from_container
2025-03-21_10-37-26  | including dev binds
2025-03-21_10-37-27  | 2025-03-21_10-37-27  | api call to [deploy_from_container] with [{}]
2025-03-21_10-37-27  | 2025-03-21_10-37-27  | deploying to [/ws]
2

In [4]:
task = agent.GenerateWorkflow(
    given=[xgdb, refdb],
    transforms=[transforms],
    targets=[
        dtypes["orf_annotations"].WithLineage([dtypes["contigs"]]),
    ]
)
for step in task.plan.steps:
    print(f">>> {step.transform.name}")
    for x in step.uses:
        print(x.path)
    print("---")
    for x in step.produces:
        print(x.path)
    print()

>>> prodigal
fosmid.fna
prodigal.oci.uri
---
orfs.faa

>>> blast
orfs.faa
swissprot_bcaa.faa
blast.oci.uri
---
annotations.csv



In [5]:
agent.StageWorkflow(task, on_exist="clear")
# agent.StageWorkflow(task, on_exist="update")
# agent.StageWorkflow(task)

2025-03-21_10-37-27  | connecting to deployed agent
2025-03-21_10-37-27  | starting relay service


E| > 2025-03-21_10-37-28 E| relay server already running in [relay/connections]


 | > 2025-03-21_10-37-28  | connecting to relay as [hadCHVhO4x4l]
2025-03-21_10-37-28 W| task already staged at [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/O7u9btAO]
2025-03-21_10-37-28 W| clearing previously staged task
2025-03-21_10-37-28  | sending metadata for workflow [O7u9btAO]
2025-03-21_10-37-31  | staging
 | > including dev binds
 | > 2025-03-21_10-37-32  | api call to [stage_workflow] with [{'task_key': 'O7u9btAO'}]
 | > 2025-03-21_10-37-32  | staging workflow [O7u9btAO] with [2] data libs and [1] transform libs
 | > 2025-03-21_10-37-32  | ex| /home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home
 | > 2025-03-21_10-37-32  | work [/ws/runs/O7u9btAO]
 | > 2025-03-21_10-37-32  | data [/msm_home/data]
 | > 2025-03-21_10-37-32  | external work [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local_home/runs/O7u9btAO]
 | > 2025-03-21_10-37-32  | external data [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/local

In [6]:
agent.RunWorkflow(task)

2025-03-21_10-37-32  | connecting to deployed agent
2025-03-21_10-37-33  | starting relay service


E| > 2025-03-21_10-37-33 E| relay server already running in [relay/connections]


 | > 2025-03-21_10-37-33  | connecting to relay as [gywZTkQAI0wL]
2025-03-21_10-37-33  | executing workflow [O7u9btAO]
2025-03-21_10-37-33  | closing connection


In [8]:
agent.CheckWorkflow(task)

2025-03-21_10-37-50  | connecting to deployed agent
2025-03-21_10-37-51  | starting relay service
 | > 2025-03-21_10-37-51  | connecting to relay as [q4I0KNUjSkMk]


E| > 2025-03-21_10-37-51 E| relay server already running in [relay/connections]


 | > including dev binds
 | > 2025-03-21_10-37-52  | api call to [check_workflow] with [{'key': 'O7u9btAO'}]
 | > 2025-03-21_10-37-52  | searching for logs
 | > 2025-03-21_10-37-52  | found [1] runs
 | > 2025-03-21_10-37-52  |     1: [logs.2025-03-21_10-37-33]
 | > 2025-03-21_10-37-52  | here is the main log of the latest run [logs.2025-03-21_10-37-33]
 | > 2025-03-21_10-37-52  | >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
 | > 2025-03-21_10-37-52  | 
 | > including dev binds
 | > 2025-03-21_10-37-34  | api call to [run_workflow] with [{'key': 'O7u9btAO', 'log_dir': '_metasmith/logs.2025-03-21_10-37-33'}]
 | > 2025-03-21_10-37-34  | start time [2025-03-21_10-37-34]
 | > 2025-03-21_10-37-34  | running workflow [O7u9btAO] with preset [default]
 | > 2025-03-21_10-37-34  | loading agent metadata
 | > 2025-03-21_10-37-34  | workspace [/msm_home/runs/O7u9btAO]
 | > 2025-03-21_10-37-34  | external workspace [/home/tony/workspace/tools/Metasmith/main/local_mock/cache/loca